# **Simulating federated learning with shares**

In [1]:
import torch
import numpy as np

import utils
from client import Client
from aggregator import Aggregator

from importlib import reload
reload(utils)

DATA_PATH = "../../02_data/mnist"
NUM_CLIENTS = 10
N_AGGREGATORS = 4

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [2]:
#loading test data:
test_images, test_labels = np.load(f"{DATA_PATH}/test_images.npy"), np.load(f"{DATA_PATH}/test_labels.npy")
test_labels = test_labels.astype(int)
test_data_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(
        torch.tensor(test_images.reshape(-1, 1, 28, 28))/255.0,
        torch.nn.functional.one_hot(torch.tensor(test_labels), num_classes=10).type(torch.float32)),
    batch_size=512, shuffle=False)

## Testing with a single client

In [3]:
client_train_images, client_train_labels = np.load(f"{DATA_PATH}/client_data/client_0_train_images.npy"), np.load(f"{DATA_PATH}/client_data/client_0_train_labels.npy")
client_train_labels = client_train_labels.astype(int)
client_data_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(
        torch.tensor(client_train_images.reshape(-1, 1, 28, 28))/255.0,
        torch.nn.functional.one_hot(torch.tensor(client_train_labels), num_classes=10).type(torch.float32)),
    batch_size=512, shuffle=True)

client = Client("client_0", N_AGGREGATORS,
                client_data_loader, device=device)

In [4]:
train_loss = client.train(epochs=1)
test_loss = utils.test(client.model, device, test_data_loader)
print(f"Client 0 train loss: {train_loss}, test loss: {test_loss}")
client.save_shares("client_output", [f"agg_{i}" for i in range(N_AGGREGATORS)])

Client 0 train loss: [np.float64(2.2913149992624917)], test loss: 2.2752777465820313


### Check shares

If everything is fine, test loss of shares shall be higher than the test loss of the client itself.

In [5]:
models = []
for a_i in range(N_AGGREGATORS):
    model = utils.load_model(f"client_output/share_client_0_agg_{a_i}.pth", device)
    test_loss = utils.test(model, device, test_data_loader)
    print(f"Share {a_i} test loss: {test_loss}")
    models.append(model)

Share 0 test loss: 2.277032116699219
Share 1 test loss: 2.2749157836914065
Share 2 test loss: 2.276177355957031
Share 3 test loss: 2.2762267639160156


It's seems fine.

---

If everything is fine, after FedAvg, we shall get back the client's loss itself.

In [6]:
reload(utils)

fed_model = utils.fed_avg(models, [len(client_data_loader.dataset)]*N_AGGREGATORS)
test_loss = utils.test(fed_model, device, test_data_loader)
print(f"FedAvg model test loss: {test_loss}")

FedAvg model test loss: 2.275277734375


It also looks nice.

### Aggregation architecture

In [7]:
aggregators = [] #needs to be topologically ordered!
for a_i in range(N_AGGREGATORS):
    agg = Aggregator(aggregator_id=f"agg_{a_i}", client_list=[f"client_0"],
                                client_data_sizes = [len(client_data_loader.dataset)])
    agg.aggregate(f"client_output")
    agg.save_model(f"client_output", f"agg_top")
    aggregators.append(agg)

top_agg = Aggregator(aggregator_id="agg_top",
                                client_list=[f"agg_{a_i}" for a_i in range(N_AGGREGATORS)],
                                client_data_sizes = [len(client_data_loader.dataset)]*N_AGGREGATORS)
top_agg.aggregate(f"client_output")
top_agg.save_model(f"client_output", f"federated_model")


Again, we shall get the original loss:

In [8]:
fed_model = utils.load_model(f"client_output/share_agg_top_federated_model.pth", device)
test_loss = utils.test(fed_model, device, test_data_loader)
print(f"FedAvg model test loss: {test_loss}")

FedAvg model test loss: 2.275277734375


Seems working.

## Simulation

10 clients --shares--> 4 bottom-level aggregators -> top-level aggregator

In [9]:
COM_ROUNDS = 3

In [10]:
#initializing clients:
clients = []
client_data_loaders = []
for i in range(NUM_CLIENTS):
    client_train_images, client_train_labels = np.load(f"{DATA_PATH}/client_data/client_{i}_train_images.npy"), np.load(f"{DATA_PATH}/client_data/client_{i}_train_labels.npy")
    client_train_labels = client_train_labels.astype(int)
    client_data_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(
            torch.tensor(client_train_images.reshape(-1, 1, 28, 28))/255.0,
            torch.nn.functional.one_hot(torch.tensor(client_train_labels), num_classes=10).type(torch.float32)),
        batch_size=512, shuffle=True)
    client_data_loaders.append(client_data_loader)
    clients.append(Client(client_id=f"client_{i}",
                                 n_aggregators=N_AGGREGATORS,
                                 device = device,
                                 train_data_loader=client_data_loader))

In [11]:
aggregators = [] #needs to be topologically ordered!
for a_i in range(N_AGGREGATORS):
    agg = Aggregator(aggregator_id=f"agg_{a_i}", client_list=[f"client_{i}" for i in range(NUM_CLIENTS)],
                                client_data_sizes = [len(client_data_loaders[i].dataset) for i in range(NUM_CLIENTS)])
    aggregators.append(agg)
    

top_agg = Aggregator(aggregator_id="agg_top",
                                client_list=[f"agg_{a_i}" for a_i in range(N_AGGREGATORS)],
                                client_data_sizes = [np.sum([len(client_data_loaders[i].dataset) for i in range(NUM_CLIENTS)])]*N_AGGREGATORS)
top_agg.aggregate(f"client_output")
top_agg.save_model(f"client_output", f"federated_model")

In [12]:
for round in range(COM_ROUNDS):
    #model download:
    for client in clients:
        client.model = utils.load_model(f"client_output/share_agg_top_federated_model.pth")
        
    print(f"Communication round {round+1}/{COM_ROUNDS}")

    #local training:
    for client in clients:
        train_loss = client.train(epochs=1)
        test_loss = utils.test(client.model, device, test_data_loader)
        print(f"Client {client.client_id} train loss: {train_loss}, test loss: {test_loss}")
        client.save_shares("client_output", [f"agg_{i}" for i in range(N_AGGREGATORS)])
    
    #aggregation:
    for agg in aggregators:
        agg.aggregate(f"client_output")
        agg.save_model(f"client_output", f"agg_top")
    
    top_agg.aggregate(f"client_output")
    top_agg.save_model(f"client_output", f"federated_model")


Communication round 1/3
Client client_0 train loss: [np.float64(2.280509114265442)], test loss: 2.275277685546875
Client client_1 train loss: [np.float64(2.279413421948751)], test loss: 2.275277685546875
Client client_2 train loss: [np.float64(2.2782938877741494)], test loss: 2.275277685546875
Client client_3 train loss: [np.float64(2.2791648308436074)], test loss: 2.275277685546875
Client client_4 train loss: [np.float64(2.280214707056681)], test loss: 2.275277685546875
Client client_5 train loss: [np.float64(2.2786526878674827)], test loss: 2.275277685546875
Client client_6 train loss: [np.float64(2.280185103416443)], test loss: 2.275277685546875
Client client_7 train loss: [np.float64(2.2789523800214133)], test loss: 2.275277685546875
Client client_8 train loss: [np.float64(2.280966858069102)], test loss: 2.275277685546875
Client client_9 train loss: [np.float64(2.280008316040039)], test loss: 2.275277685546875
Communication round 2/3
Client client_0 train loss: [np.float64(2.279832